<strong> **Concept: JSON — Your Agent's Data Exchange Format**</strong>
<h6>
JSON is a text format for data exchange. Python's json module gives you four functions: dumps/loads for strings, dump/load for files. dump writes to a file, load reads from a file, both use with open(...). Types JSON doesn't know about — like NumPy floats and ints — require a custom encoder that extends json.JSONEncoder and overrides default. The encoder checks isinstance and converts unknown types to native Python types; unknown of the unknown delegate to the parent class.
</h6>


**<b>JSON (JavaScript Object Notation) </b> **

<h6> is the universal language for data exchange. 
Every API you'll call — OpenAI, Anthropic, FastAPI services, LangChain tools — speaks JSON. 

If you can read and write JSON fluently, you can talk to anything. 

<b><u>Part 1: What JSON Looks Like </u></b>

JSON is a text format for data. It looks almost like Python dictionaries and lists, but with stricter rules.

In [ ]:
# json

{
  "name": "sales_q4",
  "rows": 250000,
  "columns": 18,
  "clean": True,
  "null_rates": {"revenue": 0.02, "country": 0.0}
}

<h6>
<b> Rules: </b>

- Keys must be strings in double quotes.

- Strings must use double quotes (not single).

- Values can be: string, number, boolean (true/false), null, object ({}), or array ([]).

- No trailing commas.

- No comments.

That's it. Simple, strict, and universal.

<b><u> Part 2: Python ↔ JSON Mapping</u> </b>

When Python converts between the two, it maps types like this:

| Python       | JSON       |
|--------------|------------|
| `dict`        | object `{}` |
| `list`, `tuple` | array `[]` |
| `str`         | string `""` |
| `int`, `float` | number     |
| `True` / `False` | `true` / `false` |
| `None`        | `null`     |

JSON has no concept of:

- datetime

- Custom class instances

- bytes

- set

- numpy types

* ! These will raise TypeError unless you handle them — which is the second half of the concept.

<b><u> Part 3: The Four Functions</u></b>

The json module gives you four functions. The names differ by one letter:

| Function              | Input / Output                  | Use                     |
|-----------------------|---------------------------------|-------------------------|
| `json.dumps(obj)`     | Python object → string          | When you need a string  |
| `json.loads(s)`       | String → Python object          | When you have a string  |
| `json.dump(obj, f)`   | Python object → file            | When you write to a file|
| `json.load(f)`        | File → Python object             | When you read from a file|

Mnemonic:

- <b>dumps / loads</b> end in s → work with strings.

- <b> dump / load (no s) </b>→ work with file objects.

- One letter, completely different interface.



<b><u> Part 4: Line-by-Line — Serialization </u></b>

In [ ]:
#python
import json

profile_data = {
    "name": "sales_q4",
    "rows": 250000,
    "columns": 18,
    "clean": True,
    "null_rates": {"revenue": 0.02, "country": 0.0}
}
# A regular Python dict. Values are Python types.

json_string = json.dumps(profile_data, indent=2)

# json.dumps → "dump string."
# Converts the dict into a JSON-formatted string.
# indent=2 makes it pretty-printed with 2-space indentation.
# Without indent, it produces a single line: {"name": "sales_q4", ...}.

In [ ]:
# Result

{
  "name": "sales_q4",
  "rows": 250000,
  "columns": 18,
  "clean": true,
  "null_rates": {
    "revenue": 0.02,
    "country": 0.0
  }
}

# Notice: True became true, and the layout is nested.

<b><u> Part 5: Line-by-Line — Deserialization</u><b>

In [ ]:
# python
loaded = json.loads(json_string)
# json.loads → "load string."
# Parses the JSON string back into a Python dict.
# loaded is now a Python dict, exactly like the original.

print(loaded["null_rates"]["revenue"])   # 0.02
# Access works exactly like a Python dict, because it is a Python dict.
# Nested access works the same way.

<b><u> Part 6: Writing and Reading Files</u></b>

In [ ]:
with open("profile.json", "w") as f:
    json.dump(profile_data, f, indent=2)
    
# open(..., "w") — open for writing.
# json.dump(obj, f, indent=2) — write the JSON directly to the file object.
# No s on dump → it writes to a file, not returns a string.

with open("profile.json", "r") as f:
    loaded = json.load(f)
    
# open(..., "r") — open for reading.
# json.load(f) — parse the file object as JSON, return a Python object.
# No s on load → it reads from a file, not a string.
# This pattern — with open(...) as f: json.dump/load(...) — is the standard way to persist Python data as JSON.

<b><u>Part 7: Why You Can't Always Serialize</u></b>

<h6>JSON only knows about the primitive types listed in Part 2. If you try to serialize something else, you get a TypeError.</h6>

In [ ]:
import json
import numpy as np

data = {"mean": np.float64(142.5)}
json.dumps(data)
# TypeError: Object of type float64 is not JSON serializable

<b>Why?</b>

<h6> np.float64 looks like a float but is not Python's built-in float. 

The JSON encoder checks :

- isinstance(obj, (dict, list, str, int, float, bool, NoneType)) and np.float64 fails every check.

You'll hit this constantly in data/ML code. Pandas values, NumPy values, custom class instances, datetimes — none of them serialize by default.</h6>

<b><u> Part 8: The Custom Encoder </u></b>

<h6> The fix is a custom encoder: a subclass of json.JSONEncoder that knows how to handle the types JSON doesn't.</h6>


In [ ]:
# python
class AgentEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.integer):
            return int(obj)
        return super().default(obj)

In [ ]:
# Line by line Explanation:
# python
class AgentEncoder(json.JSONEncoder):
# Subclass Python's built-in JSONEncoder.

# python
    def default(self, obj):
# default is called by json.dumps when it encounters a type it doesn't know.
# It receives the problematic object as obj.

        if isinstance(obj, np.floating):
            return float(obj)
# np.floating is the base class for all NumPy float types (np.float64, np.float32, etc.).
# Converts the NumPy float to a native Python float, which JSON can serialize.

# python
        if isinstance(obj, np.integer):
            return int(obj)
# Same idea, for NumPy ints.

# python
        return super().default(obj)
    
# If the object isn't one we know, delegate to the parent class's default.
# The parent will raise TypeError, which is the default behavior — you don't silence errors you don't understand.

In [ ]:
# Now:
# python
json.dumps(data, cls=AgentEncoder)
# cls=AgentEncoder tells json.dumps to use your encoder instead of the default.
# The encoder pattern is extensible. Whenever you hit a new unsupported type, 
# add an if isinstance(...) branch. Datetimes, sets, custom classes — same pattern.

<b><u>Part 9: The Design Pattern — Extending default</u></b>

<h6>
Why <b><i> super().default(obj) </i></b> instead of just raising an error yourself?

Because the parent class already knows how to raise the correct TypeError with a helpful message like:

* text

<b><i>
TypeError: Object of type Decimal is not JSON serializable </i></b>

If you raise your own error, you'd lose that message. Delegating to the parent keeps the standard behavior for unknown types.

<b>
Rule: handle what you know; delegate everything else.</b>
</h6>

Part 10: Common Pitfalls

Pitfall | What goes wrong

Using dumps when you meant dump |You get a string instead of writing to the file

Using load when you meant loads | You pass a string to a function that expects a file object — TypeError

Forgetting indent=2	| Output is one long line — hard to read

Not handling NumPy/Pandas types	| TypeError at serialization time

Using single quotes in JSON by hand	| JSON requires double quotes

Adding trailing commas	| JSON forbids them

Storing datetime without conversion | TypeError — JSON has no date type 

</h6>
